In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline, interp1d
import plotly.graph_objects as go
from scipy.signal import correlate, find_peaks, butter, filtfilt, welch
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from plotly.subplots import make_subplots
import plotly.subplots as sp
import math
from scipy.integrate import cumulative_trapezoid
import os

In [2]:
# Caminhos
file_path_sp = "Inputs/10_lento(sl).csv"
file_path_vicon = "Inputs/10_lento(vc).xlsx"

# Carregar dados
df_sp = pd.read_csv(file_path_sp) #Aceleration Smartphone 
df_vc = pd.read_excel(file_path_vicon, skiprows=3)

# Definir parâmetros
fs = 60 # Frequência de amostragem (Hz)
dt = 1 / fs

# Saída
Sujeito = '03' 
velocidade = 'rapido' # Definir a VELOCIDADE do sujeito
file_output = 'Outputs/' # definir a PASTA de saída dos arquivos

In [ ]:
# =====================================================
# 1. Aceleração do Esterno - Smartphone - Interpolação
# =====================================================
Time_sp = df_sp["accelerometerTimestamp_sinceReboot(s)"].values
Signal_sp = df_sp["accelerometerAccelerationY(G)"].values * -9.80665
Signal_sp -= 9.80665
Time_diff_sp = np.diff(Time_sp)
Time_diff_sp = np.insert(Time_diff_sp, 0, 0)
Time_cumsum_sp = np.cumsum(Time_diff_sp)
if not np.all(np.diff(Time_cumsum_sp) > 0):
    unique_indices_sp = np.unique(Time_cumsum_sp, return_index=True)[1]
    Time_cumsum_sp = Time_cumsum_sp[unique_indices_sp]
    Signal_sp = Signal_sp[unique_indices_sp]
    sorted_indices_sp = np.argsort(Time_cumsum_sp)
    Time_cumsum_sp = Time_cumsum_sp[sorted_indices_sp]
    Signal_sp = Signal_sp[sorted_indices_sp]
New_Time_sp = np.arange(Time_cumsum_sp.min(), Time_cumsum_sp.max(), dt)
cs_sp = CubicSpline(Time_cumsum_sp, Signal_sp)
Signal_interpolated_sp = cs_sp(New_Time_sp)

# ========================================
# 2. Deslocamento do Esterno - Vicon - Interpolação
# ========================================
df_vc.columns = ['Frame', 'Time', 'Esterno_X', 'Esterno_Y', 'Esterno_Z', 'C7_X', 'C7_Y', 'C7_Z']
df_vc['Time'] = pd.to_numeric(df_vc['Time'], errors='coerce')
df_vc['Esterno_Z'] = pd.to_numeric(df_vc['Esterno_Z'], errors='coerce')
df_vc.dropna(subset=['Time', 'Esterno_Z'], inplace=True)
Time_vc = df_vc['Time'].to_numpy()
Signal_vc = df_vc['Esterno_Z'].to_numpy()
Time_diff_vc = np.diff(Time_vc)
Time_diff_vc = np.insert(Time_diff_vc, 0, 0)
Time_cumsum_vc = np.cumsum(Time_diff_vc)
if not np.all(np.diff(Time_cumsum_vc) > 0):
    unique_indices_vc = np.unique(Time_cumsum_vc, return_index=True)[1]
    Time_cumsum_vc = Time_cumsum_vc[unique_indices_vc]
    Signal_vc = Signal_vc[unique_indices_vc]
    sorted_indices_vc = np.argsort(Time_cumsum_vc)
    Time_cumsum_vc = Time_cumsum_vc[sorted_indices_vc]
    Signal_vc = Signal_vc[sorted_indices_vc]
New_Time_vc = np.arange(Time_cumsum_vc.min(), Time_cumsum_vc.max(), dt)
cs_linear = interp1d(Time_cumsum_vc, Signal_vc, kind='linear')
Signal_interpolated_vc = cs_linear(New_Time_vc)

# ========================================
# Plotagem lado a lado
# ========================================
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Smartphone - Aceleração Z",
    "Vicon - Deslocamento Z"
])

fig.add_trace(go.Scatter(x=Time_cumsum_sp, y=Signal_sp, mode='lines', name='Bruto Smartphone', line=dict(color='cyan')), row=1, col=1)
fig.add_trace(go.Scatter(x=New_Time_sp, y=Signal_interpolated_sp, mode='lines', name='Interpolado Smartphone', line=dict(color='red')), row=1, col=1)

fig.add_trace(go.Scatter(x=Time_cumsum_vc, y=Signal_vc, mode='lines', name='Bruto Vicon', line=dict(color='cyan')), row=1, col=2)
fig.add_trace(go.Scatter(x=New_Time_vc, y=Signal_interpolated_vc, mode='lines', name='Interpolado Vicon', line=dict(color='red')), row=1, col=2)

fig.update_layout(
    title="Comparação dos Sinais - Esterno Z",
    width=1500,
    height=400,
    template="plotly_dark",
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(title='Tempo (s)', showgrid=True, gridcolor='gray'),
    yaxis=dict(title='Sinal', showgrid=True, gridcolor='gray')
)

fig.show()

In [4]:
#=================== Dados Smartphone ===================#

#------------------ Normalizar ao redor de zero ------------------#
Signal_interpolated_sp -= np.mean(Signal_interpolated_sp)

# ========================================
# FFT global da Aceleração - Smartphone
# ========================================

# Parâmetros
n = len(Signal_interpolated_sp)
freqs = np.fft.rfftfreq(n, d=dt)  # Frequências positivas
fft_values = np.fft.rfft(Signal_interpolated_sp)  # FFT
fft_magnitude = np.abs(fft_values)

# Encontrar pico principal de frequência
peak_freq = freqs[np.argmax(fft_magnitude)]
print(f"Frequência dominante (pico FFT) do sinal de aceleração do smartphone: {peak_freq:.2f} Hz")

# ========================================
# Análise por Janelas da Frequência Dominante
# ========================================

# Parâmetros da janela
window_duration = 5  # em segundos
step_duration = 1    # sobreposição (em segundos)
window_size = int(window_duration * fs)
step_size = int(step_duration * fs)

dominant_frequencies = []
window_times = []

for start in range(0, len(Signal_interpolated_sp) - window_size, step_size):
    end = start + window_size
    window_signal = Signal_interpolated_sp[start:end]
    window_signal -= np.mean(window_signal)  # remover offset

    window_fft = np.fft.rfft(window_signal)
    window_freqs = np.fft.rfftfreq(len(window_signal), d=dt)
    window_magnitude = np.abs(window_fft)

    # ignorar o 0 Hz (offset)
    valid_freqs = window_freqs[1:]
    valid_magnitude = window_magnitude[1:]

    peak_freq_window = valid_freqs[np.argmax(valid_magnitude)]
    dominant_frequencies.append(peak_freq_window)

    # tempo central da janela
    time_center = New_Time_sp[start + window_size // 2]
    window_times.append(time_center)

# ========================================
# Gráficos Lado a Lado com make_subplots
# ========================================

fig_combined = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'FFT Global da Aceleração do Smartphone',
        'Frequência Dominante por Janela'
    ]
)

# FFT global
fig_combined.add_trace(go.Scatter(
    x=freqs, y=fft_magnitude,
    mode='lines',
    line=dict(color='lime'),
    name='FFT Global'
), row=1, col=1)

# Frequência dominante por janela
fig_combined.add_trace(go.Scatter(
    x=window_times,
    y=dominant_frequencies,
    mode='lines+markers',
    line=dict(color='orange'),
    name='Frequência por Janela'
), row=1, col=2)

# Layout
fig_combined.update_layout(
    title='Análise de Frequência - FFT Global vs. Por Janela',
    template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    width=1400,
    height=500
)

fig_combined.update_xaxes(title_text='Frequência (Hz)', row=1, col=1)
fig_combined.update_yaxes(title_text='Magnitude', row=1, col=1)
fig_combined.update_xaxes(title_text='Tempo (s)', row=1, col=2)
fig_combined.update_yaxes(title_text='Frequência Dominante (Hz)', row=1, col=2)

fig_combined.show()

Frequência dominante (pico FFT) do sinal de aceleração do smartphone: 0.02 Hz


In [5]:
# === PARÂMETROS DO FILTRO ===
fc = 1  # Frequência de corte (Hz) – ajuste conforme necessário
order = 2  # Ordem do filtro

# === FUNÇÃO DE FILTRAGEM BUTTERWORTH ===
def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyq = 0.5 * fs  # Frequência de Nyquist
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# === APLICA O FILTRO AO SINAL INTERPOLADO ===
Signal_filtered_sp = butter_lowpass_filter(Signal_interpolated_sp, fc, fs, order)

Signal_filtered_sp -= np.mean(Signal_filtered_sp)
Signal_interpolated_sp -= np.mean(Signal_interpolated_sp)


# === PLOTAGEM DO SINAL BRUTO vs FILTRADO ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=New_Time_sp,
    y=Signal_interpolated_sp,
    mode='lines',
    name='Interpolado Bruto',
    line=dict(color='magenta')
))

fig.add_trace(go.Scatter(
    x=New_Time_sp,
    y=Signal_filtered_sp,
    mode='lines',
    name='Filtrado (Butterworth)',
    line=dict(color='lime')
))

fig.update_layout(
    title="Filtro no Sinal Interpolado do Smartphone (Esterno - Z)",
    xaxis_title="Tempo (s)",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500,
    height=400,
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99)
)

fig.show()


In [6]:
# === ZERO CROSSINGS INTERPOLADOS ===
zc_indices = np.where(np.diff(np.sign(Signal_filtered_sp)))[0]
zc_times = []
zc_values = []

for i in zc_indices:
    x1, x2 = New_Time_sp[i], New_Time_sp[i+1]
    y1, y2 = Signal_filtered_sp[i], Signal_filtered_sp[i+1]
    if y2 - y1 != 0:
        zero_time = x1 - y1 * (x2 - x1) / (y2 - y1)  # interpolação linear
        zc_times.append(zero_time)
        zc_values.append(0.0)

# === ENCONTRAR PICO/VALE ENTRE CROSSINGS COM ALTURA MÍNIMA ===
altura_minima = 0.2
picos_indices = []
vales_indices = []

zc_all = [0] + [np.searchsorted(New_Time_sp, t) for t in zc_times] + [len(Signal_filtered_sp)-1]

for i in range(len(zc_all)-1):
    ini = zc_all[i]
    fim = zc_all[i+1]
    seg = Signal_filtered_sp[ini:fim+1]

    if len(seg) > 0:
        max_idx = np.argmax(seg)
        if seg[max_idx] > altura_minima:
            picos_indices.append(ini + max_idx)
        min_idx = np.argmin(seg)
        if seg[min_idx] < -altura_minima:
            vales_indices.append(ini + min_idx)

# === PLOTAGEM ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=New_Time_sp,
    y=Signal_filtered_sp,
    mode='lines',
    name='Filtrado (Butterworth)',
    line=dict(color='lime')
))

fig.add_trace(go.Scatter(
    x=New_Time_sp[picos_indices],
    y=Signal_filtered_sp[picos_indices],
    mode='markers',
    name='Picos',
    marker=dict(color='red', size=9, symbol='circle')
))

fig.add_trace(go.Scatter(
    x=New_Time_sp[vales_indices],
    y=Signal_filtered_sp[vales_indices],
    mode='markers',
    name='Vales',
    marker=dict(color='cyan', size=9, symbol='circle')
))

fig.add_trace(go.Scatter(
    x=zc_times,
    y=zc_values,
    mode='markers',
    name='Zero Crossings',
    marker=dict(color='yellow', size=7, symbol='circle-open')
))

fig.update_layout(
    title="Sinal Filtrado com Picos, Vales e Zero Crossings ",
    xaxis_title="Tempo (s)",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500,
    height=400,
    font=dict(color='white'),
    plot_bgcolor='black',
    paper_bgcolor='black',
    legend=dict(
        x=1.02,
        y=1,
        xanchor='left',
        yanchor='top',
        bordercolor='gray',
        borderwidth=1
    )
)

fig.show()


In [7]:
# === 1. Tempo do segundo pico ===
tempo_primeiro_pico = New_Time_sp[picos_indices[1]]

# === 2. Último zero crossing anterior ao primeiro pico ===
zc_tempos_np = np.array(zc_times)
zc_anteriores = zc_tempos_np[zc_tempos_np < tempo_primeiro_pico]
if len(zc_anteriores) == 0:
    raise ValueError("Não há zero crossing anterior ao primeiro pico.")
tempo_inicio = zc_anteriores[-1]

# === 3. Tempo final (30 segundos após o zero crossing) ===
tempo_fim = min(tempo_inicio + 31.5, New_Time_sp[-1])

# === 4. Máscara do corte
mask_corte = (New_Time_sp >= tempo_inicio) & (New_Time_sp <= tempo_fim)
New_Time_cortado = New_Time_sp[mask_corte] - tempo_inicio  # zerar tempo
Signal_cortado = Signal_filtered_sp[mask_corte]

# === 5. Atualizar eventos (picos, vales e zero crossings) dentro do intervalo
def eventos_dentro_intervalo(indices, tempo_total):
    return [i for i in indices if tempo_total[i] >= tempo_inicio and tempo_total[i] <= tempo_fim]

# Pegar eventos válidos
picos_cortados = eventos_dentro_intervalo(picos_indices, New_Time_sp)
vales_cortados = eventos_dentro_intervalo(vales_indices, New_Time_sp)
zc_cortados = [i for i, t in enumerate(zc_times) if tempo_inicio <= t <= tempo_fim]

# Ajustar tempos para o novo eixo
picos_tempo = [New_Time_sp[i] - tempo_inicio for i in picos_cortados]
vales_tempo = [New_Time_sp[i] - tempo_inicio for i in vales_cortados]
zc_tempo = [zc_times[i] - tempo_inicio for i in zc_cortados]

# === 6. Plotar
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=New_Time_cortado,
    y=Signal_cortado,
    mode='lines',
    name='Sinal Cortado (30s)',
    line=dict(color='deepskyblue')
))

fig.add_trace(go.Scatter(
    x=picos_tempo,
    y=[Signal_filtered_sp[i] for i in picos_cortados],
    mode='markers',
    name='Picos',
    marker=dict(color='red', size=9, symbol='circle')
))

fig.add_trace(go.Scatter(
    x=vales_tempo,
    y=[Signal_filtered_sp[i] for i in vales_cortados],
    mode='markers',
    name='Vales',
    marker=dict(color='blue', size=9, symbol='circle')
))

fig.add_trace(go.Scatter(
    x=zc_tempo,
    y=[0]*len(zc_tempo),
    mode='markers',
    name='Zero Crossings',
    marker=dict(color='yellow', size=7, symbol='circle-open')
))

fig.update_layout(
    title="Sinal Cortado com Tempo Zerado (30s) + Eventos",
    xaxis_title="Tempo (s) desde o início do corte",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500,
    height=400,
    font=dict(color='white'),
    plot_bgcolor='black',
    paper_bgcolor='black',
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top', bordercolor='gray', borderwidth=1)
)

fig.show()


In [ ]:
#=================== Dados Smartphone ===================#

# ------------------ Normalizar ao redor de zero ------------------
Signal_cortado -= np.mean(Signal_cortado)

# ========================================
# FFT do Sinal Cortado - Smartphone
# ========================================

# Parâmetros
n = len(Signal_cortado)
dt = 1 / fs
freqs = np.fft.rfftfreq(n, d=dt)  # Frequências positivas
fft_values = np.fft.rfft(Signal_cortado)
fft_magnitude = np.abs(fft_values)

# Encontrar pico principal de frequência
peak_freq2 = freqs[np.argmax(fft_magnitude)]
print(f"Frequência dominante (pico FFT) do sinal de aceleração do smartphone (corte): {peak_freq2:.2f} Hz")

# ========================================
# Análise por Janelas da Frequência Dominante no Sinal Cortado
# ========================================

# Parâmetros da janela
window_duration = 10  # em segundos
step_duration = 1    # sobreposição (em segundos)
window_size = int(window_duration * fs)
step_size = int(step_duration * fs)

dominant_frequencies = []
window_times = []

for start in range(0, len(Signal_cortado) - window_size, step_size):
    end = start + window_size
    window_signal = Signal_cortado[start:end]
    window_signal -= np.mean(window_signal)  # remover offset

    window_fft = np.fft.rfft(window_signal)
    window_freqs = np.fft.rfftfreq(len(window_signal), d=dt)
    window_magnitude = np.abs(window_fft)

    # ignorar o 0 Hz (offset)
    valid_freqs = window_freqs[1:]
    valid_magnitude = window_magnitude[1:]

    peak_freq_window = valid_freqs[np.argmax(valid_magnitude)]
    dominant_frequencies.append(peak_freq_window)

    # tempo central da janela
    time_center = New_Time_cortado[start + window_size // 2]
    window_times.append(time_center)

# ========================================
# Gráficos Lado a Lado com make_subplots
# ========================================

fig_combined = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'FFT do Sinal Cortado - Smartphone',
        'Frequência Dominante por Janela (Sinal Cortado)'
    ]
)

# FFT global
fig_combined.add_trace(go.Scatter(
    x=freqs, y=fft_magnitude,
    mode='lines',
    line=dict(color='lime'),
    name='FFT (Sinal Cortado)'
), row=1, col=1)

# Frequência dominante por janela
fig_combined.add_trace(go.Scatter(
    x=window_times,
    y=dominant_frequencies,
    mode='lines+markers',
    line=dict(color='orange'),
    name='Frequência por Janela (Corte)'
), row=1, col=2)

# Layout
fig_combined.update_layout(
    title='Análise de Frequência - FFT (Corte) vs. Por Janela',
    template='plotly_dark',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    width=1400,
    height=500
)

fig_combined.update_xaxes(title_text='Frequência (Hz)', row=1, col=1)
fig_combined.update_yaxes(title_text='Magnitude', row=1, col=1)
fig_combined.update_xaxes(title_text='Tempo (s)', row=1, col=2)
fig_combined.update_yaxes(title_text='Frequência Dominante (Hz)', row=1, col=2)

fig_combined.show()


Frequência dominante (pico FFT) do sinal de aceleração do smartphone (corte): 0.51 Hz


In [9]:
# === PARÂMETROS DO FILTRO ===
nyq_f = 0.7832 - 0.1054 * peak_freq - 0.2219 * peak_freq2
fc_f = 0.4082 - 0.0955 * peak_freq + 0.1417 * peak_freq2

# Limitar faixa segura de fc e Nyq para evitar distorções extremas
fc_f = max(0.2, min(fc_f, 1.2))     # fc entre 0.2 e 1.2 Hz
nyq_f = max(0.3, min(nyq_f, 1.5))   # Nyquist factor entre 0.3 e 1.5

# === Função para aplicar filtro passa-alta Butterworth ===
def highpass_filter(signal, fs, fc=0.5, order=2):
    nyq = nyq_f * fs
    normal_cutoff = fc / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)

# === 1. Extrair o sinal bruto no intervalo do corte ===
acel_bruta_cortada = Signal_interpolated_sp[mask_corte]

# === 2. Remover a média local da aceleração
acel_corrigida = acel_bruta_cortada - np.mean(acel_bruta_cortada)

# === 3. Integrar para obter velocidade
velocidade_cortada = cumulative_trapezoid(acel_corrigida, New_Time_cortado, initial=0)

# === 4. Aplicar filtro passa-alta à velocidade
fs_local = 1 / np.mean(np.diff(New_Time_cortado))  # frequência amostral
velocidade_corrigida = highpass_filter(velocidade_cortada, fs=fs_local, fc=fc_f, order=2)

# === 1. Integrar a velocidade corrigida para obter deslocamento ===
deslocamento_corrigido = cumulative_trapezoid(velocidade_corrigida, New_Time_cortado, initial=0)

# === 2. Zerar deslocamento inicial
deslocamento_corrigido -= deslocamento_corrigido[0]

# === 1. Interpolar deslocamento do Vicon no tempo do smartphone + converter mm → cm ===
deslocamento_vicon_cortado = cs_linear(New_Time_cortado) / 10  # mm → cm

# === 2. Converter deslocamento do smartphone de m → cm ===
deslocamento_corrigido_cm = deslocamento_corrigido * 100  # m → cm

# === 3. Correlação cruzada para alinhar no tempo ===
corr = correlate(deslocamento_corrigido_cm - np.mean(deslocamento_corrigido_cm),
                 deslocamento_vicon_cortado - np.mean(deslocamento_vicon_cortado), mode='full')

lag_idx = np.argmax(corr) - (len(deslocamento_corrigido_cm) - 1)

# === 4. Aplicar lag e alinhar ===
if lag_idx > 0:
    deslocamento_corrigido_alinhado = deslocamento_corrigido_cm[lag_idx:]
    deslocamento_vicon_alinhado = deslocamento_vicon_cortado[:len(deslocamento_corrigido_alinhado)]
    tempo_alinhado = New_Time_cortado[:len(deslocamento_corrigido_alinhado)]
else:
    deslocamento_vicon_alinhado = deslocamento_vicon_cortado[-lag_idx:]
    deslocamento_corrigido_alinhado = deslocamento_corrigido_cm[:len(deslocamento_vicon_alinhado)]
    tempo_alinhado = New_Time_cortado[:len(deslocamento_vicon_alinhado)]

# === 5. Centralizar ambos os sinais (remover offset) ===
deslocamento_corrigido_alinhado -= np.mean(deslocamento_corrigido_alinhado)
deslocamento_vicon_alinhado -= np.mean(deslocamento_vicon_alinhado)

# === 6. Plotar os deslocamentos corrigidos e alinhados ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=tempo_alinhado,
    y=deslocamento_corrigido_alinhado,
    mode='lines',
    name='Deslocamento Smartphone (Corrigido e Alinhado)',
    line=dict(color='lime')
))

fig.add_trace(go.Scatter(
    x=tempo_alinhado,
    y=deslocamento_vicon_alinhado,
    mode='lines',
    name='Deslocamento Vicon (Interpolado e Alinhado)',
    line=dict(color='orange')
))

fig.update_layout(
    title="Deslocamento Corrigido e Alinhado - Smartphone vs Vicon",
    xaxis_title="Tempo (s)",
    yaxis_title="Deslocamento (cm)",
    template="plotly_dark",
    width=1500,
    height=400,
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top', bordercolor='gray', borderwidth=1)
)  
fig.show()

# === 7. RMS entre os deslocamentos corrigidos e alinhados ===
rms = np.sqrt(np.sum((deslocamento_corrigido_alinhado - deslocamento_vicon_alinhado) ** 2)/ len(deslocamento_corrigido_alinhado))
print(f"RMS entre deslocamento corrigido do smartphone e deslocamento do Vicon: {rms:.2f} cm")

# === 8. Amplitude do deslocamento do Vicon ===
amplitude_vicon = np.max(deslocamento_vicon_alinhado) - np.min(deslocamento_vicon_alinhado)

# === 9. Erro percentual entre os deslocamentos corrigidos e alinhados ===
error_pct = (rms / amplitude_vicon) * 100.0
print(f"Erro percentual entre deslocamento corrigido do smartphone e deslocamento do Vicon: {error_pct:.2f}%")


RMS entre deslocamento corrigido do smartphone e deslocamento do Vicon: 0.55 cm
Erro percentual entre deslocamento corrigido do smartphone e deslocamento do Vicon: 6.70%
